# データセットアップ
Databricks Unity Catalog上にカタログ・スキーマ・Volume・テーブルを作成し、PDFドキュメントを解析してナレッジベースを構築します。

## 1. 設定読み込み

In [ ]:
%run ./_config

## 2. カタログ / スキーマ / Volume 作成

In [ ]:
import sys
major, minor = sys.version_info[:2]
assert (major, minor) >= (3, 11), f"Python 3.11以上が必要です。現在: {major}.{minor}。DBR 15.4以上をご利用ください。"

In [ ]:
# カタログ作成
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
if current_catalog != catalog:
    catalogs = [r['catalog'] for r in spark.sql("SHOW CATALOGS").collect()]
    if catalog not in catalogs:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")

spark.sql(f"USE CATALOG `{catalog}`")

# スキーマ作成
spark.sql(f"CREATE DATABASE IF NOT EXISTS `{schema}`")
spark.sql(f"USE `{catalog}`.`{schema}`")

# Volume作成
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

print(f"環境準備完了: `{catalog}`.`{schema}`")
print(f"Volume: /Volumes/{catalog}/{schema}/{volume_name}")

## 3. テーブル作成・データ投入
架空の電子機器メーカー「TechNova」のカスタマーサポートデータを生成します。

In [ ]:
%pip install faker -q
dbutils.library.restartPython()

In [ ]:
%run ./_config

In [ ]:
import random
from datetime import date, timedelta
from pyspark.sql import Row
from faker import Faker

fake_ja = Faker('ja_JP')
Faker.seed(42)
random.seed(42)

spark.sql(f"USE `{catalog}`.`{schema}`")

In [ ]:
# 製品マスタ（日本語）
products_data_ja = [
    {"product_id": 1, "product_name": "SmartWatch X100", "category": "スマートウォッチ", "price": 39800, "description": "高精度心拍センサー搭載。防水IP68対応。バッテリー持続7日間。"},
    {"product_id": 2, "product_name": "SmartWatch X200", "category": "スマートウォッチ", "price": 59800, "description": "GPS内蔵、血中酸素測定対応。有機ELディスプレイ。バッテリー持続5日間。"},
    {"product_id": 3, "product_name": "NoiseGuard Pro", "category": "ヘッドフォン", "price": 34800, "description": "アクティブノイズキャンセリング搭載。最大30時間再生。ハイレゾ対応。"},
    {"product_id": 4, "product_name": "NoiseGuard Lite", "category": "ヘッドフォン", "price": 19800, "description": "軽量設計のワイヤレスヘッドフォン。最大20時間再生。折りたたみ可能。"},
    {"product_id": 5, "product_name": "SmartSpeaker Echo", "category": "スマートスピーカー", "price": 14800, "description": "360度サウンド対応。音声アシスタント搭載。スマートホーム連携。"},
    {"product_id": 6, "product_name": "SmartSpeaker Echo Plus", "category": "スマートスピーカー", "price": 24800, "description": "高音質Dolby Atmos対応。マルチルーム再生。タッチスクリーン付き。"},
    {"product_id": 7, "product_name": "TechTab S10", "category": "タブレット", "price": 49800, "description": "10.5インチ有機EL。Snapdragon搭載。ペン入力対応。バッテリー12時間。"},
    {"product_id": 8, "product_name": "TechTab S12", "category": "タブレット", "price": 69800, "description": "12.4インチ大画面。キーボード付属。マルチタスク最適化。5G対応。"},
    {"product_id": 9, "product_name": "FitBand Z1", "category": "フィットネスバンド", "price": 9800, "description": "歩数・心拍・睡眠追跡。防水IP67。バッテリー14日間。"},
    {"product_id": 10, "product_name": "FitBand Z2", "category": "フィットネスバンド", "price": 15800, "description": "GPS内蔵フィットネスバンド。ストレス測定。バッテリー10日間。"},
    {"product_id": 11, "product_name": "AirBuds Pro", "category": "イヤホン", "price": 24800, "description": "完全ワイヤレス。ANC搭載。IPX5防水。最大8時間再生。"},
    {"product_id": 12, "product_name": "AirBuds Lite", "category": "イヤホン", "price": 12800, "description": "軽量ワイヤレスイヤホン。最大6時間再生。タッチ操作対応。"},
    {"product_id": 13, "product_name": "PowerBank Ultra", "category": "アクセサリー", "price": 7800, "description": "20000mAh大容量。USB-C/A対応。急速充電対応。"},
    {"product_id": 14, "product_name": "SmartScale Body", "category": "スマートスケール", "price": 8800, "description": "体重・体脂肪・筋肉量・水分量測定。アプリ連携。最大8人登録。"},
    {"product_id": 15, "product_name": "HomeHub Mini", "category": "スマートホーム", "price": 6800, "description": "スマートホームハブ。Wi-Fi/Bluetooth/Zigbee対応。音声操作対応。"},
    {"product_id": 16, "product_name": "SmartCam Indoor", "category": "スマートカメラ", "price": 11800, "description": "1080p室内カメラ。動体検知。双方向通話。クラウド録画対応。"},
    {"product_id": 17, "product_name": "SmartCam Outdoor", "category": "スマートカメラ", "price": 18800, "description": "4K屋外カメラ。IP66防水。暗視機能。AI人物検知。"},
    {"product_id": 18, "product_name": "ChargePad Duo", "category": "アクセサリー", "price": 5800, "description": "デュアルワイヤレス充電パッド。Qi対応。最大15W急速充電。"},
    {"product_id": 19, "product_name": "SmartThermo", "category": "スマートホーム", "price": 4800, "description": "スマート温湿度計。アプリ通知。履歴グラフ表示。"},
    {"product_id": 20, "product_name": "TechNova Stylus", "category": "アクセサリー", "price": 9800, "description": "4096段階筆圧。傾き検知。TechTab専用スタイラスペン。"},
]

products_ja_df = spark.createDataFrame([Row(**p) for p in products_data_ja])
products_ja_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.products")
print(f"products テーブル作成完了")

In [ ]:
# 顧客データ（日本語）
customers_ja = []
for i in range(200):
    customers_ja.append({
        "customer_id": 10001 + i,
        "first_name": fake_ja.first_name(),
        "last_name": fake_ja.last_name(),
        "email": fake_ja.email(),
        "phone": fake_ja.phone_number(),
        "address": fake_ja.address().replace("\n", " "),
        "prefecture": fake_ja.prefecture(),
        "registration_date": str(fake_ja.date_between(start_date='-3y', end_date='today')),
        "customer_status": random.choices(["アクティブ", "休止", "解約"], weights=[0.85, 0.10, 0.05])[0],
        "loyalty_tier": random.choices(["ブロンズ", "シルバー", "ゴールド", "プラチナ"], weights=[0.4, 0.3, 0.2, 0.1])[0],
    })

customers_ja_df = spark.createDataFrame([Row(**c) for c in customers_ja])
customers_ja_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.customers")
print(f"customers テーブル作成完了")

In [ ]:
# 注文データ（日本語）
orders_ja = []
order_statuses_ja = ["配送済み", "配送中", "処理中", "キャンセル", "返品"]
for i in range(500):
    cid = random.choice([c["customer_id"] for c in customers_ja])
    pid = random.choice(products_data_ja)
    qty = random.randint(1, 3)
    orders_ja.append({
        "order_id": 50001 + i,
        "customer_id": cid,
        "product_id": pid["product_id"],
        "product_name": pid["product_name"],
        "quantity": qty,
        "total_amount": pid["price"] * qty,
        "order_date": str(fake_ja.date_between(start_date='-1y', end_date='today')),
        "status": random.choices(order_statuses_ja, weights=[0.60, 0.15, 0.10, 0.10, 0.05])[0],
    })

orders_ja_df = spark.createDataFrame([Row(**o) for o in orders_ja])
orders_ja_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.orders")
print(f"orders テーブル作成完了")

In [ ]:
# サポートチケット（日本語）
ticket_categories_ja = ["返品・交換", "技術サポート", "製品問合せ", "配送問題", "アカウント"]
ticket_priorities_ja = ["低", "中", "高", "緊急"]
ticket_statuses_ja = ["オープン", "対応中", "解決済み", "クローズ"]

tickets_ja = []
for i in range(300):
    cid = random.choice([c["customer_id"] for c in customers_ja])
    pid = random.choice(products_data_ja)
    cat = random.choices(ticket_categories_ja, weights=[0.25, 0.30, 0.20, 0.15, 0.10])[0]

    descriptions_ja = {
        "返品・交換": f"{pid['product_name']}を購入しましたが、{random.choice(['イメージと違った', '初期不良がある', 'サイズが合わない', '別のモデルに変更したい'])}ため、{random.choice(['返品', '交換'])}を希望します。",
        "技術サポート": f"{pid['product_name']}について、{random.choice(['電源が入らない', 'Bluetooth接続ができない', 'アプリと同期できない', '充電ができない', 'フリーズする'])}問題が発生しています。",
        "製品問合せ": f"{pid['product_name']}の{random.choice(['スペック詳細', '対応機種', '保証期間', '付属品', '在庫状況'])}について教えてください。",
        "配送問題": f"注文した{pid['product_name']}が{random.choice(['届かない', '破損していた', '違う商品が届いた', '配送が遅延している'])}。",
        "アカウント": f"{random.choice(['パスワードを忘れた', 'メールアドレスを変更したい', '退会方法を知りたい', 'ポイント残高を確認したい'])}。",
    }

    tickets_ja.append({
        "ticket_id": 80001 + i,
        "customer_id": cid,
        "product_name": pid["product_name"],
        "category": cat,
        "priority": random.choices(ticket_priorities_ja, weights=[0.20, 0.40, 0.30, 0.10])[0],
        "status": random.choices(ticket_statuses_ja, weights=[0.20, 0.30, 0.35, 0.15])[0],
        "description": descriptions_ja[cat],
        "created_date": str(fake_ja.date_between(start_date='-6m', end_date='today')),
    })

tickets_ja_df = spark.createDataFrame([Row(**t) for t in tickets_ja])
tickets_ja_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.support_tickets")
print(f"support_tickets テーブル作成完了")

In [ ]:
# ポリシー（日本語）
policies_ja = [
    {"policy_id": 1, "policy_name": "返品ポリシー", "category": "返品・交換",
     "content": "商品到着後14日以内であれば返品可能です。未開封・未使用の商品に限ります。開封済みの場合は初期不良のみ対応します。返品送料はお客様負担ですが、初期不良の場合は当社負担となります。返金は返品商品の確認後、5営業日以内に元の支払い方法に返金されます。"},
    {"policy_id": 2, "policy_name": "交換ポリシー", "category": "返品・交換",
     "content": "商品到着後30日以内であれば同一商品との交換が可能です。異なる商品への交換は、返品後に新規注文をお願いします。交換商品の在庫がない場合は返金対応となります。交換送料は初期不良の場合は当社負担、お客様都合の場合はお客様負担です。"},
    {"policy_id": 3, "policy_name": "保証ポリシー", "category": "保証",
     "content": "TechNova製品は購入日から1年間のメーカー保証が付きます。保証期間中の自然故障は無償修理または交換対応します。水没・落下・改造による故障は保証対象外です。ゴールド/プラチナ会員は保証期間が2年間に延長されます。修理期間は通常7-14営業日です。"},
    {"policy_id": 4, "policy_name": "技術サポートポリシー", "category": "サポート",
     "content": "技術サポートは営業時間内（平日9:00-18:00）に対応します。メール・チャット・電話でのお問い合わせが可能です。初期設定・接続・アプリ連携のサポートは無償です。ハードウェア修理が必要な場合は保証ポリシーに従います。リモートサポートも提供しています。"},
    {"policy_id": 5, "policy_name": "ポイントプログラム", "category": "会員特典",
     "content": "購入金額の1%がポイントとして付与されます。シルバー会員は2%、ゴールド会員は3%、プラチナ会員は5%です。ポイントは1ポイント=1円として次回購入時に使用可能です。ポイントの有効期限は最終購入日から1年間です。年間購入10万円以上でシルバー、30万円以上でゴールド、50万円以上でプラチナに昇格します。"},
]

policies_ja_df = spark.createDataFrame([Row(**p) for p in policies_ja])
policies_ja_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.policies")
print(f"policies テーブル作成完了")

## 4. PDFをVolumeにコピー
Workspaceの `pdfs/` ディレクトリにあるPDFファイルをVolumeにコピーします。

In [ ]:
import os, shutil

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
output_dir = f"{volume_path}/pdf_documentation"
os.makedirs(output_dir, exist_ok=True)

# Workspace上のPDFディレクトリを特定
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_root = os.path.dirname(os.path.dirname(_nb_path))
    pdfs_dir = f"/Workspace{_repo_root}/pdfs"
except Exception:
    pdfs_dir = os.path.join(os.path.dirname(os.getcwd()), "pdfs")

print(f"PDF元: {pdfs_dir}")
print(f"Volume: {output_dir}")

# PDFファイルをVolumeにコピー
count = 0
for f in os.listdir(pdfs_dir):
    if f.endswith('.pdf'):
        src = os.path.join(pdfs_dir, f)
        dst = os.path.join(output_dir, f)
        shutil.copy2(src, dst)
        print(f"  コピー: {f}")
        count += 1

print(f"\n{count}件のPDFをVolumeにコピーしました")

## 5. PDF解析 → knowledge_base テーブル
`ai_parse_document` でPDFを解析し、`ai_extract` で構造化データを抽出します。

In [ ]:
# knowledge_baseテーブル作成
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog}`.`{schema}`.knowledge_base (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  product_name STRING,
  title STRING,
  content STRING,
  doc_uri STRING)
  TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print(f"knowledge_base テーブル作成完了")

In [ ]:
# ai_parse_document でPDFを解析
spark.sql(f"""
INSERT OVERWRITE TABLE `{catalog}`.`{schema}`.knowledge_base (product_name, title, content, doc_uri)
SELECT ai_extract.product_name, ai_extract.title, content, doc_uri
FROM (
  SELECT
    ai_extract(content, array('product_name', 'title')) AS ai_extract,
    content,
    doc_uri
  FROM (
    SELECT array_join(
            transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n') AS content,
           path as doc_uri
    FROM (
      SELECT ai_parse_document(content) AS parsed_document, path
      FROM READ_FILES('/Volumes/{catalog}/{schema}/{volume_name}/pdf_documentation/', format => 'binaryFile')
    )
  )
)
""")
print(f"PDF解析完了")

In [ ]:
display(spark.sql(f"SELECT id, product_name, title, length(content) as content_length, doc_uri FROM `{catalog}`.`{schema}`.knowledge_base"))

## 6. 確認
作成されたリソースを一覧表示します。

In [ ]:
print("=" * 60)
print(f"データセットアップ完了: `{catalog}`.`{schema}`")
print("=" * 60)

print(f"\n--- テーブル一覧 ---")
for table in ["products", "customers", "orders", "support_tickets", "policies", "knowledge_base"]:
    try:
        count = spark.table(f"{catalog}.{schema}.{table}").count()
        print(f"  {table}: {count}件")
    except Exception as e:
        print(f"  {table}: エラー - {e}")

print(f"\n--- Volume内容 ---")
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
for root, dirs, files in os.walk(volume_path):
    level = root.replace(volume_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")